In [1]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
# Developed by Anicet on 12/01/2023
# updated by Jingyi on 30/09/2024 (v1.1 / v1.2)
# rebuilt for the 2022 Banco de Espana IAS redesign  (v1.3)

import os
import glob
import shutil
import datetime
from time import sleep

import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By


In [2]:
#------------------------------------------------ Begin_fileName ----------------------------------------
regulatorName = 'ES BES'

print('Running {} Web Scraping Tool v.1.3'.format(regulatorName))

now = datetime.datetime.now()
processdate = now.strftime('%Y-%m-%d')
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":", ".")[:-7])

# ------ At first we will define the workspace path -----
try:
    scriptfolder = os.path.dirname(os.path.abspath(__file__))  ## production environment (.py)
except NameError:
    scriptfolder = os.getcwd()  ## notebook environment

os.chdir(scriptfolder)

tempfolder = os.path.join(scriptfolder, 'tempfolder')  # the two XLS reports land here

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)


Running ES BES Web Scraping Tool v.1.3


In [3]:
#------------------------------------------------ Begin_chromedriver ----------------------------------------
# Selenium is still required even though the payload is a plain XLS file : the IAS
# front-end mints a per-session IdUnico UUID during the handshake, and a bare
# requests.post without it is rejected with HTTP 400.

chromeOptions = webdriver.ChromeOptions()
prefs = {"plugins.always_open_pdf_externally": True,
         "download.prompt_for_download": False,
         "download.default_directory": tempfolder,
         "selectedDestinationId": tempfolder,
         "safebrowsing.enabled": True,
         "version": 2}
chromeOptions.add_experimental_option("prefs", prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.set_page_load_timeout(180)
driver.maximize_window()


In [4]:
#------------------------------------------------ Begin_Dictionnary ----------------------------------------
ARRANQUE = 'http://app.bde.es/ren_www/ren_wwwias/xml/Arranque.html'

# ListNr -> the entity-type code as published in the report files.
# ConEstab (entities WITH an establishment in Spain) keys off column COD_TIPO,
# SinEstab (entities operating WITHOUT an establishment) keys off column TIPO_ENTIDAD.
regdict = {
    'ES BES 1': 'BP',
    'ES BES 2': 'CA',
    'ES BES 3': 'CC',
    'ES BES 4': 'CO',
    'ES BES 5': 'EDE',
    'ES BES 6': 'EP',
    'ES BES 7': 'EFC',
    'ES BES 8': 'OR',
    'ES BES 10': 'SGR',
    'ES BES 11': 'SR',
    'ES BES 12': 'ST',
    'ES BES 14': 'SECC',
    'ES BES 15': 'SECE',
    'ES BES 16': 'SEDC',
    'ES BES 17': 'SEPC',
    'ES BES 18': 'ECVM',
    'ES BES 19': '21',
    'ES BES 20': '11.2',
    'ES BES 22': '19',
}

# ListNames verbatim from the Jira ticket DECD-6467.
Typology = {
    'ES BES 1': 'BANCOS',
    'ES BES 2': 'CAJAS DE AHORROS',
    'ES BES 3': 'COOPERATIVAS DE CREDITO',
    'ES BES 4': 'CREDITO OFICIAL',
    'ES BES 5': 'ENTIDADES DE DINERO ELECTRONICO',
    'ES BES 6': 'ENTIDADES DE PAGO',
    'ES BES 7': 'ESTABLECIMIENTOS FINANCIEROS DE CREDITO',
    'ES BES 8': 'OFICINAS DE REPRESENTACION EN ESPANA DE ENTIDADES DE CREDITO EXTRANJERAS',
    'ES BES 10': 'SOCIEDADES DE GARANTIA RECIPROCA',
    'ES BES 11': 'SOCIEDADES DE REAFIANZAMIENTO',
    'ES BES 12': 'SOCIEDADES DE TASACION',
    'ES BES 14': 'SUCURSALES DE ENTIDADES DE CREDITO  EXTRANJERAS COMUNITARIAS',
    'ES BES 15': 'SUCURSALES DE ENTIDADES DE CREDITO  EXTRANJERAS EXTRACOMUNITARIAS',
    'ES BES 16': 'SUCURSALES DE ENTIDADES DE DINERO ELECTRONICO EXTRANJERAS COMUNITARIAS',
    'ES BES 17': 'SUCURSALES DE ENTIDADES DE PAGO EXTRANJERAS COMUNITARIAS',
    'ES BES 18': 'TITULARES DE ESTABLECIMIENTOS DE COMPRA Y VENTA DE MONEDA EXTRANJERA',
    'ES BES 19': 'ENT. CTO. COMUN. OPER. ESPANA SIN ESTAB. (ART. 39 DIR. 2013/36/CE)',
    'ES BES 20': 'ENT. CTO. EXTRACOM. OPER. ESPANA SIN ESTAB. (ART. 6, LEY 10/2014)',
    'ES BES 22': 'ENT. FINAN., FILIALES ENT. CTO. COMUN., OPER. ESPANA SIN ESTAB. (ART. 34)',
}

# ListLabel : 1 = bank, 2 = insurance, 3 = bank & insurance, 4 = everything else.
# Banco de Espana does not supervise insurance at all (that is the DGSFP), so no 2 or 3
# appears here. Deposit-taking credit institutions, their foreign branches and the
# representative offices of foreign credit institutions -> 1; everything else -> 4.
ListLabel = {
    'ES BES 1': 1,    # BANCOS
    'ES BES 2': 1,    # CAJAS DE AHORROS
    'ES BES 3': 1,    # COOPERATIVAS DE CREDITO
    'ES BES 4': 1,    # CREDITO OFICIAL (ICO)
    'ES BES 5': 4,    # ENTIDADES DE DINERO ELECTRONICO
    'ES BES 6': 4,    # ENTIDADES DE PAGO
    'ES BES 7': 4,    # ESTABLECIMIENTOS FINANCIEROS DE CREDITO (lost credit-institution status in 2014)
    'ES BES 8': 1,    # OFICINAS DE REPRESENTACION de entidades de credito extranjeras
    'ES BES 10': 4,   # SOCIEDADES DE GARANTIA RECIPROCA
    'ES BES 11': 4,   # SOCIEDADES DE REAFIANZAMIENTO
    'ES BES 12': 4,   # SOCIEDADES DE TASACION
    'ES BES 14': 1,   # SUCURSALES de entidades de credito comunitarias
    'ES BES 15': 1,   # SUCURSALES de entidades de credito extracomunitarias
    'ES BES 16': 4,   # SUCURSALES de entidades de dinero electronico
    'ES BES 17': 4,   # SUCURSALES de entidades de pago
    'ES BES 18': 4,   # COMPRA Y VENTA DE MONEDA EXTRANJERA
    'ES BES 19': 1,   # ENT. CTO. COMUNITARIAS sin establecimiento
    'ES BES 20': 1,   # ENT. CTO. EXTRACOMUNITARIAS sin establecimiento
    'ES BES 22': 4,   # ENT. FINANCIERAS filiales de ent. cto. comunitarias
}

sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [],
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [],
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [],
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [],
          'Phone - Mother company': []}


In [5]:
#------------------------------------------------ Begin_Fonction ----------------------------------------

def bourange_same_length_array(sqldict):
    """Pad every column of sqldict out to the length of ListProcessDate."""
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key] = sqldict[key] + empty
    return sqldict


def clean(value):
    """Normalise a cell coming out of the XLS : NaN / 'nan' / whitespace -> ''."""
    if value is None:
        return ''
    text = str(value).strip()
    if text.lower() in ('nan', 'nat', 'none'):
        return ''
    return ' '.join(text.split())


def esdate(value):
    """BdE publishes dates as dd/mm/yyyy or as an Excel serial - normalise to yyyy-mm-dd."""
    text = clean(value)
    if not text:
        return ''
    for fmt in ('%d/%m/%Y', '%Y-%m-%d %H:%M:%S', '%Y-%m-%d'):
        try:
            return datetime.datetime.strptime(text, fmt).strftime('%Y-%m-%d')
        except ValueError:
            continue
    return text


def build_address(row):
    """SIGLAVIA + NOMBREVIA + NUMEROVIA are the street parts, RESTODOM the remainder."""
    street = ' '.join(p for p in (clean(row.get('SIGLAVIA')),
                                  clean(row.get('NOMBREVIA')),
                                  clean(row.get('NUMEROVIA'))) if p)
    return street, clean(row.get('RESTODOM'))


def download_report(menu_index, expected_name, label):
    """Drive the IAS report screen and wait for its XLS to land in tempfolder.

    menu_index 1 = 'Entidades con establecimiento', 2 = 'Entidades sin establecimiento'.
    Both radio groups are mandatory - submitting without them returns
    'Error 001: Seleccionar dato' instead of a file.
    """
    print('[INFO] : - requesting the {} report'.format(label))

    driver.get(ARRANQUE)
    sleep(2)
    driver.find_element(
        By.XPATH,
        '//*[@id="AreaDeNavegacioncolMapa_0"]/ul/li[1]/ul/li[{}]'.format(menu_index)).click()
    sleep(4)

    # _GrupoRadioButton_0  = 'Actual'    ( _1 = 'Completa', which also returns cancelled entities )
    # _GrupoRadioButton0_0 = 'Entidades' ( _1 = 'Actividades' )
    for radio_id in ('_GrupoRadioButton_0', '_GrupoRadioButton0_0'):
        found = driver.find_elements(By.XPATH, '//*[contains(@id,"{}")]'.format(radio_id))
        if not found:
            raise Exception('[ERROR] : - radio {} missing on the {} screen - the IAS '
                            'form has changed again'.format(radio_id, label))
        driver.execute_script("arguments[0].click();", found[0])
        sleep(1)

    driver.find_element(By.XPATH, '//*[@id="btnBuscar"]').click()

    target = os.path.join(tempfolder, expected_name)
    for _ in range(120):
        sleep(2)
        if os.path.exists(target) and not glob.glob(os.path.join(tempfolder, '*.crdownload')):
            print('[INFO] : - downloaded {} ({:.0f} KB)'.format(expected_name,
                                                                os.path.getsize(target) / 1024))
            return target

    raise Exception('[ERROR] : - {} never arrived in tempfolder'.format(expected_name))


In [6]:
#------------------------------------------------ Begin_Main ----------------------------------------
# The two search screens no longer paginate a grid with a per-entity detail page - since the
# 2022 redesign each one emits one JasperReports XLS covering every entity type at once.
# So we download twice and group by entity-type column instead of looping 19 x N pages.

con_path = download_report(1, 'informe_EntConEstab.xls', 'entidades CON establecimiento')
sin_path = download_report(2, 'informe_EntSinEstab.xls', 'entidades SIN establecimiento')

driver.quit()

# BIFF8 : openpyxl cannot read these, xlrd is required.
con = pd.read_excel(con_path, engine='xlrd', dtype=str)
sin = pd.read_excel(sin_path, engine='xlrd', dtype=str)

print('[INFO] : - con establecimiento : {} rows x {} cols'.format(*con.shape))
print('[INFO] : - sin establecimiento : {} rows x {} cols'.format(*sin.shape))

con_groups = {code: part for code, part in con.groupby('COD_TIPO')}
sin_groups = {code: part for code, part in sin.groupby('TIPO_ENTIDAD')}

# Entity types present in the source but not requested by the ticket are reported and
# skipped rather than silently dropped.
unmapped = (set(con_groups) | set(sin_groups)) - set(regdict.values())
if unmapped:
    skipped = sum(len(con_groups.get(u, [])) + len(sin_groups.get(u, [])) for u in unmapped)
    print('[WARN] : - {} row(s) in entity types not covered by the Jira ticket, skipped : {}'
          .format(skipped, ', '.join(sorted(unmapped))))

for reg in regdict:

    code = regdict[reg]
    listcode = reg.split()[-1]
    with_estab = code in con_groups

    part = con_groups.get(code, sin_groups.get(code))
    if part is None or len(part) == 0:
        print('[WARN] : - {} ({}) returned 0 rows'.format(reg, code))
        continue

    print('[INFO] : - {} | {} | {} rows'.format(reg, code, len(part)))

    for _, row in part.iterrows():

        if with_estab:
            # NOMBRE105 is the full legal name, NOMBRE50 the truncated one.
            name = clean(row.get('NOMBRE105')) or clean(row.get('NOMBRE50'))
            if not name:
                continue

            street, rest = build_address(row)

            sqldict['Name'].append(name)
            sqldict['InternalID_1'].append(clean(row.get('COD_BE')))
            sqldict['InternalID_1_type'].append('BE Code')
            sqldict['InternalID_2'].append(clean(row.get('CODIGOCIF')))
            sqldict['InternalID_2_type'].append('N.I.F')
            sqldict['LEI Code'].append(clean(row.get('CODIGO_LEI')))
            sqldict['Address_1'].append(street)
            sqldict['Address_2'].append(rest)
            sqldict['City'].append(clean(row.get('POBLACION')))
            sqldict['Zip'].append(clean(row.get('CODPOSTAL')))
            sqldict['Cntry'].append(clean(row.get('CODPAIS')) or 'ES')
            sqldict['Phone'].append(clean(row.get('TELEFONO')))
            sqldict['Fax'].append(clean(row.get('NUMFAX')))
            sqldict['Website'].append(clean(row.get('DIRINTERNET')))
            sqldict['CoType'].append(clean(row.get('NOMCOMERCIAL')))
            sqldict['RegulationDate'].append(esdate(row.get('FECHAALTA')))
            sqldict['CancellationDate'].append(esdate(row.get('FCHBAJA')))
            sqldict['Name - Mother Company'].append(clean(row.get('NOMENTMATRIZ')))
            sqldict['Address_1 - Mother company'].append(clean(row.get('DIRENTMATRIZ')))
            sqldict['City - Mother company'].append(clean(row.get('LOCENTMATRIZ')))
            sqldict['Zip - Mother company'].append(clean(row.get('CPENTMATRIZ')))
            sqldict['Cntry - Mother company'].append(clean(row.get('CODPAISMATRIZ')))
            sqldict['Phone - Mother company'].append(clean(row.get('TELENTMATRIZ')))

        else:
            # The SinEstab report is much thinner - name, home country and parent only.
            name = clean(row.get('NOMBRE1'))
            if not name:
                continue

            sqldict['Name'].append(name)
            sqldict['Cntry'].append(clean(row.get('COD_PAIS')))
            sqldict['RegulationDate'].append(esdate(row.get('FECHAALTA')))
            sqldict['CancellationDate'].append(esdate(row.get('FCHBAJA')))
            sqldict['Name - Mother Company'].append(clean(row.get('NOMBRE_ENTIDAD_MATRIZ')))

        # ----- common columns -----
        sqldict['Typology'].append(Typology[reg])
        sqldict['License_Type'].append(code)
        sqldict['RegulationType'].append('Regulated')
        sqldict['RegCtry'].append('ES')
        sqldict['RegCode'].append('BES')
        sqldict['ListCode'].append(listcode)
        sqldict['ListName'].append(Typology[reg])
        sqldict['ListLabel'].append(ListLabel[reg])
        sqldict['ListLanguage'].append('ES')
        sqldict['ListProcessDate'].append(processdate)

        sqldict = bourange_same_length_array(sqldict)


[INFO] : - requesting the entidades CON establecimiento report


[INFO] : - downloaded informe_EntConEstab.xls (340 KB)
[INFO] : - requesting the entidades SIN establecimiento report


[INFO] : - downloaded informe_EntSinEstab.xls (138 KB)


[INFO] : - con establecimiento : 421 rows x 49 cols
[INFO] : - sin establecimiento : 689 rows x 10 cols
[WARN] : - 37 row(s) in entity types not covered by the Jira ticket, skipped : EFEP, EPEX, PSIC, SEFC, SFC, SFMC
[INFO] : - ES BES 1 | BP | 42 rows
[INFO] : - ES BES 2 | CA | 2 rows
[INFO] : - ES BES 3 | CC | 60 rows
[INFO] : - ES BES 4 | CO | 1 rows
[INFO] : - ES BES 5 | EDE | 12 rows
[INFO] : - ES BES 6 | EP | 51 rows
[INFO] : - ES BES 7 | EFC | 22 rows
[INFO] : - ES BES 8 | OR | 29 rows
[INFO] : - ES BES 10 | SGR | 17 rows
[INFO] : - ES BES 11 | SR | 1 rows
[INFO] : - ES BES 12 | ST | 28 rows
[INFO] : - ES BES 14 | SECC | 78 rows
[INFO] : - ES BES 15 | SECE | 3 rows
[INFO] : - ES BES 16 | SEDC | 10 rows
[INFO] : - ES BES 17 | SEPC | 11 rows
[INFO] : - ES BES 18 | ECVM | 17 rows
[INFO] : - ES BES 19 | 21 | 666 rows
[INFO] : - ES BES 20 | 11.2 | 3 rows
[INFO] : - ES BES 22 | 19 | 20 rows


In [7]:
#------------------------------------------------ Begin_writer and save df to excel ----------------------------------------
os.chdir(scriptfolder)

df = pd.DataFrame(sqldict)

df = df[df['Name'] != '']
df = df.drop_duplicates(subset=['Name', 'ListCode'], keep='first')
df = df.reset_index(drop=True)

df.to_excel(os.path.join(scriptfolder, filename), sheet_name='SQL Ready', index=False)

print('Saved {} rows to {}'.format(len(df), os.path.join(scriptfolder, filename)))
print(df.groupby('ListCode').size())

# the two XLS reports are intermediates - do not leave them behind
shutil.rmtree(tempfolder, ignore_errors=True)


Saved 1073 rows to /Users/wuj1/Library/CloudStorage/OneDrive-Moody's/Desktop/Regulator/ES BES/ES BES SQL Ready 2026-07-29 10.34.57.xlsx
ListCode
1      42
10     17
11      1
12     28
14     78
15      3
16     10
17     11
18     17
19    666
2       2
20      3
22     20
3      60
4       1
5      12
6      51
7      22
8      29
dtype: int64
